In [19]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
import os
import pandas as pd

In [4]:
base = os.path.join(os.getcwd(),'..')

In [5]:
shapefile_path = os.path.join(os.getcwd(),'..','shape','States_shapefile.shp')

In [9]:
outPath = os.path.join(base,'tiffs','stats(maize)')
if not os.path.exists(outPath):
    os.mkdir(outPath)

In [12]:
raster = os.path.join(base,'tiffs')
r1 = os.path.join(raster,'test.tiff')
agbm = os.path.join(raster,'aboveground_biomassc')

In [24]:
def get_zonal_stats(r1,r2,shapefile_path,fn,b1,b2):    
    raster1_path = r1 
    raster2_path = r2 
    shapefile_path = shapefile_path
    output_csv = fn
    
    band_raster1 = b1
    band_raster2 = b2
    
    gdf = gpd.read_file(shapefile_path)
    
    with rasterio.open(raster1_path) as src1, rasterio.open(raster2_path) as src2:
    
        if gdf.crs != src1.crs:
            gdf = gdf.to_crs(src1.crs)
    
        results = []
    
        for idx, row in gdf.iterrows():
            geom = [row.geometry]
            name = row['State_Name']
            out1, _ = mask(src1, geom, crop=True)
            band1 = out1[band_raster1 - 1].astype("float32")
    
            out2, _ = mask(src2, geom, crop=True)
            band2 = out2[band_raster2 - 1].astype("float32")
    
            nodata1 = src1.nodata
            nodata2 = src2.nodata
            if nodata1 is not None and nodata2 is not None:
                mask_nodata = (band1 == nodata1) | (band2 == nodata2)
            else:
                mask_nodata = np.zeros(band1.shape, dtype=bool)
    
            mult = band1 * band2
            mult[mask_nodata] = np.nan
            # mult = np.where(mult==0,np.nan,mult)
    
            stats = {
                "zone_id": name,
                "mean": np.nanmean(mult),
                "sum": np.nansum(mult),
                "min": np.nanmin(mult),
                "max": np.nanmax(mult),
                "count": np.sum(~np.isnan(mult))
            }
    
            results.append(stats)
    
    df = pd.DataFrame(results)
    
    # df = pd.concat([gdf.reset_index(drop=True), df], axis=1)
    
    df.to_csv(output_csv, index=False)
    


In [25]:
for r,d,f in os.walk(agbm):
    for fl in f:
        if fl.endswith('.tif'):
            name = fl.split('_')[-1].split('.')[0]
            fn = os.path.join(r,fl)
            fo = os.path.join(outPath,'Maize_rainfed_agbm'+name+'.csv')
            crop = 2
            agbm_c = 2
            get_zonal_stats(r1,fn,shapefile_path,fo,crop,agbm_c)